In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import numpy as np
import joblib
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
import warnings

warnings.filterwarnings('ignore')


class WaterQualityApp:

    def __init__(self, root):

        self.root = root
        self.root.title("Water Quality Prediction System")
        self.root.geometry("1500x850")
        self.root.configure(bg="#f4f6f7")

        self.center_window()

        # =========================
        # LOAD MODEL
        # =========================
        self.load_model()

        # =========================
        # THRESHOLDS
        # =========================
        self.THRESHOLDS = {
            'aluminium': 2.8,
            'ammonia': 32.5,
            'arsenic': 0.01,
            'barium': 2.0,
            'cadmium': 0.005,
            'chloramine': 4.0,
            'chromium': 0.1,
            'copper': 1.3,
            'flouride': 1.5,
            'bacteria': 0.0,
            'viruses': 0.0,
            'lead': 0.015,
            'nitrates': 10.0,
            'nitrites': 1.0,
            'mercury': 0.002,
            'perchlorate': 56.0,
            'radium': 5.0,
            'selenium': 0.5,
            'silver': 0.1,
            'uranium': 0.3
        }

        self.FEATURES = list(self.THRESHOLDS.keys())

        self.input_vars = {}

        self.setup_gui()

    # =========================================================
    # CENTER WINDOW
    # =========================================================
    def center_window(self):

        self.root.update_idletasks()

        width = 1500
        height = 850

        x = (self.root.winfo_screenwidth() // 2) - (width // 2)
        y = (self.root.winfo_screenheight() // 2) - (height // 2)

        self.root.geometry(f"{width}x{height}+{x}+{y}")

    # =========================================================
    # LOAD MODEL
    # =========================================================
    def load_model(self):

        try:
            self.model = joblib.load("water_quality_model.pkl")
            print("Model Loaded Successfully")

        except:
            self.model = None
            print("Model Not Found")

    # =========================================================
    # GUI
    # =========================================================
    def setup_gui(self):

        # =========================
        # HEADER
        # =========================
        header = tk.Frame(self.root, bg="#1f618d", height=80)
        header.pack(fill="x")

        title = tk.Label(
            header,
            text="💧 WATER QUALITY PREDICTION SYSTEM",
            font=("Arial", 24, "bold"),
            bg="#1f618d",
            fg="white"
        )

        title.pack(pady=20)

        # =========================
        # MAIN CONTAINER
        # =========================
        container = tk.Frame(self.root, bg="#f4f6f7")
        container.pack(fill="both", expand=True, padx=10, pady=10)

        container.grid_columnconfigure(0, weight=1)
        container.grid_columnconfigure(1, weight=2)

        container.grid_rowconfigure(0, weight=1)

        # =========================
        # LEFT PANEL
        # =========================
        left_panel = tk.Frame(
            container,
            bg="white",
            bd=2,
            relief="ridge"
        )

        left_panel.grid(row=0, column=0, sticky="nsew", padx=10)

        # =========================
        # RIGHT PANEL
        # =========================
        right_panel = tk.Frame(
            container,
            bg="white",
            bd=2,
            relief="ridge"
        )

        right_panel.grid(row=0, column=1, sticky="nsew", padx=10)

        # CREATE PANELS
        self.create_input_panel(left_panel)
        self.create_results_panel(right_panel)

        # =========================
        # BOTTOM
        # =========================
        bottom = tk.Frame(self.root, bg="#f4f6f7", height=70)
        bottom.pack(fill="x")

        predict_btn = tk.Button(
            bottom,
            text="🔍 PREDICT WATER QUALITY",
            command=self.predict,
            bg="#27ae60",
            fg="white",
            font=("Arial", 16, "bold"),
            padx=30,
            pady=10,
            cursor="hand2"
        )

        predict_btn.pack(pady=10)

    # =========================================================
    # INPUT PANEL
    # =========================================================
    def create_input_panel(self, parent):

        title = tk.Label(
            parent,
            text="📝 INPUT PARAMETERS",
            font=("Arial", 18, "bold"),
            bg="white",
            fg="#2c3e50"
        )

        title.pack(pady=15)

        input_frame = tk.Frame(parent, bg="white")
        input_frame.pack(fill="both", expand=True, padx=10)

        for i, feature in enumerate(self.FEATURES):

            row = i // 2
            col = i % 2

            frame = tk.Frame(input_frame, bg="white")
            frame.grid(row=row, column=col, padx=10, pady=8, sticky="w")

            label = tk.Label(
                frame,
                text=f"{feature.title()}:",
                font=("Arial", 10, "bold"),
                bg="white",
                width=12,
                anchor="w"
            )

            label.pack(side="left")

            var = tk.DoubleVar(value=0.0)

            self.input_vars[feature] = var

            entry = tk.Entry(
                frame,
                textvariable=var,
                width=10,
                font=("Arial", 10),
                bd=2
            )

            entry.pack(side="left", padx=5)

            limit = tk.Label(
                frame,
                text=f"Max:{self.THRESHOLDS[feature]}",
                font=("Arial", 8),
                bg="white",
                fg="gray"
            )

            limit.pack(side="left")

    # =========================================================
    # RESULTS PANEL
    # =========================================================
    def create_results_panel(self, parent):

        title = tk.Label(
            parent,
            text="📊 RESULTS & VISUALIZATION",
            font=("Arial", 18, "bold"),
            bg="white",
            fg="#2c3e50"
        )

        title.pack(pady=10)

        # =========================
        # RESULT TEXT
        # =========================
        self.results_text = scrolledtext.ScrolledText(
            parent,
            height=18,
            font=("Consolas", 10),
            bg="#fdfefe",
            bd=2
        )

        self.results_text.pack(fill="x", padx=10, pady=10)

        # =========================
        # FIGURE
        # =========================
        self.fig = Figure(figsize=(8, 3.5), dpi=100)

        self.canvas = FigureCanvasTkAgg(self.fig, master=parent)

        self.canvas.get_tk_widget().pack(
            fill="both",
            expand=True,
            padx=10,
            pady=10
        )

    # =========================================================
    # PREDICT
    # =========================================================
    def predict(self):

        if self.model is None:

            messagebox.showerror(
                "Error",
                "Model file not found!"
            )

            return

        try:

            values = []

            for feature in self.FEATURES:
                values.append(self.input_vars[feature].get())

            input_array = np.array(values).reshape(1, -1)

            prediction = self.model.predict(input_array)[0]

            try:
                probs = self.model.predict_proba(input_array)[0]

            except:
                probs = [0.5, 0.5]

            self.display_results(prediction, probs, values)

            self.update_chart(values)

            self.save_results_to_txt(prediction, probs, values)

        except Exception as e:

            messagebox.showerror(
                "Error",
                str(e)
            )

    # =========================================================
    # DISPLAY RESULTS
    # =========================================================
    def display_results(self, prediction, probs, values):

        self.results_text.delete("1.0", tk.END)

        self.results_text.insert(
            tk.END,
            "\n========== WATER QUALITY REPORT ==========\n\n"
        )

        if prediction == 1:

            result = "SAFE WATER"

        else:

            result = "UNSAFE WATER"

        confidence = probs[prediction] * 100

        self.results_text.insert(
            tk.END,
            f"Prediction : {result}\n"
        )

        self.results_text.insert(
            tk.END,
            f"Confidence : {confidence:.2f}%\n\n"
        )

        self.results_text.insert(
            tk.END,
            "------------------------------------------\n"
        )

        self.results_text.insert(
            tk.END,
            f"{'Parameter':<15}{'Value':<15}{'Status'}\n"
        )

        self.results_text.insert(
            tk.END,
            "------------------------------------------\n"
        )

        for i, feature in enumerate(self.FEATURES):

            value = values[i]

            threshold = self.THRESHOLDS[feature]

            status = "SAFE"

            if value > threshold:
                status = "DANGER"

            self.results_text.insert(
                tk.END,
                f"{feature:<15}{value:<15.4f}{status}\n"
            )

        self.results_text.insert(
            tk.END,
            "\n==========================================\n"
        )

        self.results_text.insert(
            tk.END,
            f"Generated : {datetime.now()}\n"
        )

    # =========================================================
    # UPDATE CHART
    # =========================================================
    def update_chart(self, values):

        self.fig.clear()

        ax = self.fig.add_subplot(111)

        thresholds = [
            self.THRESHOLDS[f]
            for f in self.FEATURES
        ]

        x = np.arange(len(self.FEATURES))

        bars = ax.bar(x, values)

        for i, bar in enumerate(bars):

            if values[i] > thresholds[i]:
                bar.set_color("red")

        ax.plot(
            x,
            thresholds,
            marker='o',
            linewidth=2
        )

        ax.set_xticks(x)

        ax.set_xticklabels(
            self.FEATURES,
            rotation=45,
            ha='right',
            fontsize=8
        )

        ax.set_title(
            "Water Parameters vs Thresholds",
            fontsize=12,
            fontweight="bold"
        )

        ax.grid(True, alpha=0.3)

        self.fig.tight_layout()

        self.canvas.draw()

    # =========================================================
    # SAVE TXT
    # =========================================================
    def save_results_to_txt(self, prediction, probs, values):

        try:

            filename = f"Water_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

            with open(filename, "w", encoding="utf-8") as file:

                file.write("WATER QUALITY REPORT\n")
                file.write("=" * 50 + "\n\n")

                if prediction == 1:
                    result = "SAFE WATER"
                else:
                    result = "UNSAFE WATER"

                confidence = probs[prediction] * 100

                file.write(f"Prediction : {result}\n")
                file.write(f"Confidence : {confidence:.2f}%\n")
                file.write(f"Date : {datetime.now()}\n\n")

                file.write("-" * 50 + "\n")

                file.write(
                    f"{'Parameter':<15}{'Value':<15}{'Status'}\n"
                )

                file.write("-" * 50 + "\n")

                for i, feature in enumerate(self.FEATURES):

                    value = values[i]

                    threshold = self.THRESHOLDS[feature]

                    status = "SAFE"

                    if value > threshold:
                        status = "DANGER"

                    file.write(
                        f"{feature:<15}{value:<15.4f}{status}\n"
                    )

                file.write("\n" + "=" * 50)

            print(f"Saved: {filename}")

        except Exception as e:

            print("TXT Save Error:", e)


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":

    root = tk.Tk()

    app = WaterQualityApp(root)

    root.mainloop()

Model Loaded Successfully
Saved: Water_Report_20260624_195058.txt
